# NLP-3: Data exploration and class-prior baseline

Run in order after placing the competition `train.csv` in `data/raw/`. Outputs are initially empty: no dataset findings are claimed. Labels: 0 = A wins, 1 = B wins, 2 = tie. This is separate from the breast-cancer/digits lab.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src/baseline.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/baseline.py").exists():
    raise RuntimeError("Open this notebook from the project root or notebooks folder.")
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from src.baseline import load_data, saved_split, main
FIGURES = ROOT / "results/figures"
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load and validate
The loader checks required columns, unique IDs, one-hot labels, and multi-turn JSON structure. Model names and IDs are not predictive features.

In [ ]:
df, fingerprint = load_data()
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("Dataset SHA256:", fingerprint)
print("Missing values:")
print(df.isna().sum())
print("Repeated prompt rows:", df.prompt_group.duplicated().sum())
print("Duplicate prompt/response triplets:", df.duplicated(["prompt_text", "response_a_text", "response_b_text"]).sum())

## 2. Class balance and text lengths
Review these outputs and write what you observe. Class imbalance can make majority-class accuracy misleading.

In [ ]:
counts = df.label.value_counts().reindex([0, 1, 2], fill_value=0)
print(pd.DataFrame({"count": counts, "percentage": counts / len(df) * 100}))
fig, ax = plt.subplots(figsize=(6, 4))
counts.rename(index={0:"A wins", 1:"B wins", 2:"Tie"}).plot.bar(ax=ax)
ax.set_ylabel("Examples")
ax.set_title("Preference class distribution")
fig.tight_layout()
fig.savefig(FIGURES / "class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
lengths = pd.DataFrame({col: df[col + "_text"].str.len() for col in ["prompt", "response_a", "response_b"]})
print(lengths.describe().round(2))
fig, ax = plt.subplots(figsize=(7, 4))
lengths.plot.hist(bins=50, alpha=0.5, ax=ax)
ax.set_xlabel("Characters")
ax.set_ylabel("Examples")
ax.set_title("Prompt and response lengths")
fig.tight_layout()
fig.savefig(FIGURES / "text_lengths.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## 3. Save reusable prompt-grouped splits
Approximately 60% training, 20% validation, 20% final local test. Grouping takes precedence over exact ratios. Existing split files are reused and checked against the dataset fingerprint. Share IDs privately with teammates; inspect additional source/duplicate-response leakage before claiming a complete audit.

In [ ]:
split = saved_split(df, fingerprint)
print(pd.crosstab(split, df.label))
print("Saved assignments:", ROOT / "data/splits/split_ids.csv")

## 4. Class-prior validation baseline
This uses training class proportions only and evaluates on validation. The final local test is not scored. It also records package versions.

In [ ]:
main()

## 5. Write the first findings

Replace the pending entries in `reports/first_draft_report.md` with actual observations:
- What are the class counts and majority fraction?
- Are any fields missing or examples duplicated?
- Can prompts or other sources leak across partitions?
- What are the validation log loss, accuracy, and macro F1?
- What does the baseline fail to capture?

Next: numerical-feature and TF-IDF logistic-regression baselines on exactly these split IDs. Keep calibration fitting separate from final local test evaluation.